# IMT Controls — Slot Filling from Knowledge Graph
## Single Round · First 14 Slots · Answerability Analysis

**Goal:** Use the knowledge graph to fill the first 14 slots of the baseline,  
then analyse which questions the graph can answer and which it structurally cannot.

**First 14 slots covered:**
- `ACH-01` (4 slots): Supplier security tracking
- `ACH-02` (4 slots): Security clauses in subcontractor contracts  
- `DATA-01` (3 slots): TLS encryption in transit  
- `DATA-02` (3 slots): Encryption at rest (first 3 slots)


## Bloc 1 — Imports & Configuration

In [ ]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import ChatNVIDIA

load_dotenv()

print("Imports OK")
print(f"NVIDIA_API_KEY : {'OK' if os.getenv('NVIDIA_API_KEY') else 'MISSING'}")


## Bloc 2 — LLM Initialization

In [ ]:
llm = ChatNVIDIA(
    model="mistralai/mistral-large-3-675b-instruct-2512",
    api_key=os.getenv("NVIDIA_API_KEY"),
    temperature=0,
    max_completion_tokens=8000,
)
print("LLM ready")


## Bloc 3 — Load Knowledge Graph

In [ ]:
# ── Update this path to your graph JSON ──────────────────────────────────────
GRAPH_JSON_PATH = "SecuraOps High_graph_4pass.json"   # <-- change as needed

with open(GRAPH_JSON_PATH, "r", encoding="utf-8") as f:
    graph = json.load(f)

entities  = graph.get("entities",      [])
relations = graph.get("relationships", [])

print(f"Graph loaded: {len(entities)} entities, {len(relations)} relationships")

from collections import Counter
label_counts = Counter(e["label"] for e in entities)
print("\nNode types present:")
for label, count in sorted(label_counts.items()):
    print(f"  {label:<20}: {count}")


## Bloc 4 — Load Baseline & Select First 14 Slots

In [ ]:
BASELINE_CSV = "Baseline_pour_IMT_Controls_Slots_IMT_.csv"

df_all = pd.read_csv(BASELINE_CSV, sep=";", encoding="utf-8-sig")

# Normalise the long description column name
desc_col = [c for c in df_all.columns if "Description" in c][0]
df_all.rename(columns={desc_col: "description"}, inplace=True)

# ── Take exactly the first 14 slots ──────────────────────────────────────────
SLOT_COUNT = 14
df_slots = df_all.head(SLOT_COUNT).reset_index(drop=True)

print(f"Total slots in baseline : {len(df_all)}")
print(f"Slots selected          : {len(df_slots)}  (first {SLOT_COUNT})")
print()
print(df_slots[["Control_ID", "Control_Name", "Slot_Key", "Slot_Type"]].to_string(index=False))


## Bloc 5 — Build Compact Graph Summary

In [ ]:
def build_graph_summary(entities: list, relationships: list) -> str:
    """
    Produce a compact, readable text representation of the knowledge graph.
    Entities are listed with all their properties.
    Relationships are listed as triples.
    """
    lines_e = []
    for e in entities:
        props = ", ".join(f'{k}="{v}"' for k, v in e.get("properties", {}).items())
        lines_e.append(f"  [{e['label']}]  {props}")

    lines_r = []
    for r in relationships:
        from_val = list(r.get("from_key", {}).values())
        to_val   = list(r.get("to_key",   {}).values())
        lines_r.append(
            f"  ({r['from_label']}:{from_val[0] if from_val else '?'})"
            f" -[{r['rel_type']}]->"
            f" ({r['to_label']}:{to_val[0] if to_val else '?'})"
        )

    return (
        "=== ENTITIES ===\n" + "\n".join(lines_e) +
        "\n\n=== RELATIONSHIPS ===\n" + "\n".join(lines_r)
    )


GRAPH_SUMMARY = build_graph_summary(entities, relations)
print(f"Graph summary : {len(GRAPH_SUMMARY):,} chars  (~{len(GRAPH_SUMMARY)//4:,} tokens estimated)")
print("\nFirst 600 chars preview:")
print(GRAPH_SUMMARY[:600])


## Bloc 6 — Build the Slot-Filling Prompt

In [ ]:
def format_slot(row) -> str:
    """Format a single slot row into a clear instruction block for the LLM."""
    stype = str(row.get("Slot_Type", "bool"))
    enum  = row.get("Enum_Values")

    # Handle NaN enum values
    try:
        import math
        if math.isnan(float(enum)):
            enum = None
    except (TypeError, ValueError):
        pass

    if stype == "bool":
        type_hint = "bool  →  true  or  false"
    elif stype == "enum" and enum:
        type_hint = f"enum  →  one of:  {enum}"
    elif stype == "list[string]":
        type_hint = 'list[string]  →  JSON array, e.g. ["AES-256", "RSA-2048"]'
    else:
        type_hint = "bool  →  true  or  false"

    return (
        f"slot_key   : \"{row['Slot_Key']}\"\n"
        f"control    : {row['Control_ID']} — {row['Control_Name']}\n"
        f"type       : {type_hint}\n"
        f"instruction: {row['description']}"
    )


SYSTEM_PROMPT = """You are a cybersecurity analyst filling a security-assessment baseline.
You are given a knowledge graph (entities + relationships extracted from supplier documents)
and a list of exactly 14 slots to fill.

Rules:
- Answer ONLY from what is explicitly present in the knowledge graph.
- For bool slots     : return true or false.
- For enum slots     : return exactly one of the listed allowed values.
- For list[string]   : return a JSON array of strings (empty [] if nothing found).
- If the graph does not contain enough information → set "value": null and "answerability": "unanswerable".
- If information is partial or ambiguous           → set "answerability": "ambiguous".
- If the graph clearly supports the answer         → set "answerability": "answerable".
- Always include a short "evidence" field (≤ 1 sentence) quoting the graph node/property used.
  If nothing was found write: "Not found in graph."
- Return ONLY a valid JSON array — no markdown, no explanation outside the JSON.

Output format (one object per slot, same order as input):
[
  {
    "slot_key"     : "<key>",
    "control_id"   : "<Control_ID>",
    "value"        : <true|false|"enum_value"|["item"]|null>,
    "answerability": "answerable" | "ambiguous" | "unanswerable",
    "evidence"     : "<one-sentence justification>"
  },
  ...
]"""

# Build the user message
slot_blocks = "\n\n".join(format_slot(row) for _, row in df_slots.iterrows())
USER_PROMPT = (
    f"KNOWLEDGE GRAPH:\n{GRAPH_SUMMARY}\n\n"
    f"SLOTS TO FILL ({SLOT_COUNT} slots):\n\n{slot_blocks}\n\n"
    "Fill all 14 slots using only the knowledge graph above. Return only the JSON array."
)

print(f"System prompt : {len(SYSTEM_PROMPT):,} chars")
print(f"User prompt   : {len(USER_PROMPT):,} chars")
print(f"Total         : {len(SYSTEM_PROMPT)+len(USER_PROMPT):,} chars  (~{(len(SYSTEM_PROMPT)+len(USER_PROMPT))//4:,} tokens estimated)")


## Bloc 7 — Call the LLM (Single Round)

In [ ]:
def call_llm(system_prompt: str, user_prompt: str) -> str:
    """Call the LLM and return raw string output."""
    messages = [
        ("system", system_prompt),
        ("human",  user_prompt),
    ]
    raw = llm.invoke(messages).content.strip()
    # Strip markdown code fences if the model added them
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
        raw = raw.strip()
    return raw


print("Calling LLM — single round, 14 slots...")
raw_output = call_llm(SYSTEM_PROMPT, USER_PROMPT)
print("LLM call complete.")
print()
print("Raw output (first 800 chars):")
print(raw_output[:800])


## Bloc 8 — Parse & Validate the Results

In [ ]:
def parse_and_validate(raw: str, df_slots: pd.DataFrame) -> pd.DataFrame:
    """
    Parse the LLM JSON output and align it with the expected slot list.
    Missing slots get value=None, answerability='error'.
    """
    try:
        results = json.loads(raw)
        if not isinstance(results, list):
            raise ValueError(f"Expected a list, got {type(results)}")
    except (json.JSONDecodeError, ValueError) as e:
        print(f"[PARSE ERROR] {e}")
        print(f"Raw output:\n{raw}")
        results = []

    # Index by slot_key for lookup
    result_map = {r.get("slot_key"): r for r in results}

    rows = []
    for _, slot in df_slots.iterrows():
        key = slot["Slot_Key"]
        r   = result_map.get(key, {})
        rows.append({
            "control_id"   : slot["Control_ID"],
            "control_name" : slot["Control_Name"],
            "slot_key"     : key,
            "slot_type"    : slot["Slot_Type"],
            "value"        : r.get("value"),
            "answerability": r.get("answerability", "error" if not r else "unanswerable"),
            "evidence"     : r.get("evidence", "Not returned by model"),
        })
    return pd.DataFrame(rows)


df_results = parse_and_validate(raw_output, df_slots)

print("\nFilled baseline — 14 slots:")
print()
for _, row in df_results.iterrows():
    icon = {"answerable": "✅", "ambiguous": "⚠️", "unanswerable": "❌", "error": "💥"}.get(
        row["answerability"], "?")
    print(f"  {icon} {row['slot_key']:<40}  value={str(row['value']):<12}  [{row['answerability']}]")


## Bloc 9 — Evidence Details

In [ ]:
print("Evidence per slot:\n")
for _, row in df_results.iterrows():
    print(f"  [{row['control_id']}] {row['slot_key']}")
    print(f"    value    : {row['value']}")
    print(f"    evidence : {row['evidence']}")
    print()


## Bloc 10 — Answerability Analysis

Which questions can the graph answer and which styles of questions it cannot — and **why**.


In [ ]:
from collections import Counter

counts = Counter(df_results["answerability"])
total  = len(df_results)

print("=" * 60)
print("ANSWERABILITY SUMMARY")
print("=" * 60)
print(f"  ✅ Answerable   : {counts.get('answerable',   0):>2}  ({counts.get('answerable',   0)/total*100:.0f}%)")
print(f"  ⚠️  Ambiguous    : {counts.get('ambiguous',    0):>2}  ({counts.get('ambiguous',    0)/total*100:.0f}%)")
print(f"  ❌ Unanswerable : {counts.get('unanswerable', 0):>2}  ({counts.get('unanswerable', 0)/total*100:.0f}%)")
print(f"  💥 Errors       : {counts.get('error',        0):>2}")
print()

print("=" * 60)
print("WHAT THE GRAPH CAN ANSWER")
print("=" * 60)
print("""
The knowledge graph is extracted from governance/policy documents.
It reliably captures:

  1. EXISTENCE of security controls (bool)
     e.g. "Does the supplier have security clauses in subcontracts?"
          → ACH-01.security_clauses_subcontracts
          → ACH-02.security_clauses
     The graph has Clause, Agreement, Supplier nodes — presence is directly
     inferable from their properties and relationships.

  2. NAMED contractual mechanisms
     e.g. "Does the supplier maintain a list of critical suppliers?"
          → ACH-01.critical_suppliers
     Supplier and Agreement nodes with named properties answer this.

  3. NAMED TECHNOLOGY values (enum / list)
     e.g. "What encryption algorithms are used?"
          → DATA-02.algorithms
     Algorithm nodes have a 'name' property extracted verbatim from docs.

  4. DOCUMENTED protocol versions (enum)
     e.g. "What is the minimum TLS version?"
          → DATA-01.tls_min_version
     Protocol nodes capture named protocols and their versions.
""")

print("=" * 60)
print("WHAT THE GRAPH CANNOT ANSWER (structural gaps)")
print("=" * 60)
print("""
  1. NEGATIVE EVIDENCE questions
     e.g. "Is an NDA the ONLY security clause used?"  → ACH-02.nda_only
     WHY: The graph captures positive assertions (what IS present).
          Confirming that nothing else exists requires reading the full
          document — absence of a node does not equal confirmed absence.

  2. INFRASTRUCTURE-LEVEL CONFIGURATIONS
     e.g. "Is HSTS enforced?"   → DATA-01.hsts
          "Is PFS enforced?"    → DATA-01.pfs
     WHY: These are web-server/TLS-stack settings, not written in governance
          documents. The graph source documents are policy/contract texts,
          not network scan reports or server config files.

  3. OPERATIONAL CADENCE / PROCESS QUALITY
     e.g. "Are encryption keys rotated regularly?" → DATA-02.key_rotation
     WHY: The graph captures WHAT is implemented, not HOW rigorously.
          Key rotation frequency is rarely stated as a named node — it
          lives in procedural text that the schema has no node type for.

  4. SCOPE ENUMERATIONS when not explicitly documented
     e.g. "What is the scope of data-at-rest encryption?" → DATA-02.scope
     WHY: The schema has no dedicated scope field on Asset/Algorithm nodes.
          Even when scope is mentioned in text, it is absorbed into a
          description string rather than a typed enum property.
""")


## Bloc 11 — Export Results

In [ ]:
# CSV
out_csv = "imt_14slots_filled.csv"
df_results.to_csv(out_csv, index=False, encoding="utf-8-sig")
print(f"Saved: {out_csv}")

# JSON
out_json = "imt_14slots_filled.json"
with open(out_json, "w", encoding="utf-8") as f:
    json.dump(df_results.to_dict(orient="records"), f, indent=2, ensure_ascii=False)
print(f"Saved: {out_json}")

# Display final table
df_results[["control_id", "slot_key", "slot_type", "value", "answerability", "evidence"]]
